In [32]:
import os, random, pandas as pd, pymongo, pprint
from dotenv import load_dotenv
from datetime import datetime, timedelta, time as dtime

load_dotenv(".env")

MONGO = {
    "host":       os.getenv("MONGO_HOST", "localhost"),
    "port":       int(os.getenv("MONGO_PORT", 27017)),
    "database":   os.getenv("MONGO_DATABASE", "logistics_db"),
    "collection": os.getenv("MONGO_COLLECTION", "delivery_events"),
}

DATA_RAW = os.path.join("..", "data", "raw")
TEST_CSV = os.path.join(DATA_RAW, "test.csv")

random.seed(42)
print("Config loaded from .env")
print(f"MongoDB : {MONGO['host']}:{MONGO['port']}/{MONGO['database']}")

Config loaded from .env
MongoDB : localhost:27017/logistics_db


In [33]:
# Load test.csv only

df = pd.read_csv(TEST_CSV)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

Shape: 11,399 rows × 19 columns


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City
0,0x2318,COIMBRES13DEL01,NaN,NaN,11.003669,76.976494,11.043669,77.016494,30-03-2022,NaN,15:05:00,conditions NaN,NaN,3,Drinks,electric_scooter,1,No,Metropolitian
1,0x3474,BANGRES15DEL01,28,4.6,12.975377,77.696664,13.085377,77.806664,29-03-2022,20:30:00,20:35:00,conditions Windy,Jam,0,Snack,motorcycle,1,No,Metropolitian
2,0x9420,JAPRES09DEL03,23,4.5,26.911378,75.789034,27.001378,75.879034,10-03-2022,19:35:00,19:45:00,conditions Stormy,Jam,0,Drinks,motorcycle,1,No,Metropolitian


In [34]:
print(df.columns.tolist())

['ID', 'Delivery_person_ID', 'Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'Order_Date', 'Time_Orderd', 'Time_Order_picked', 'Weatherconditions', 'Road_traffic_density', 'Vehicle_condition', 'Type_of_order', 'Type_of_vehicle', 'multiple_deliveries', 'Festival', 'City']


In [35]:
# Clean

def clean_df(df):
    df = df.copy()

    # Strip whitespace safely
    for col in df.columns:
        if df[col].dtype == object:
            try:
                df[col] = df[col].astype(str).str.strip()
                df[col] = df[col].replace("nan", None)
            except:
                pass

    # Fix "conditions Sunny" → "Sunny"
    df["Weatherconditions"] = df["Weatherconditions"].str.replace("conditions ", "", regex=False)

    # Coerce numerics
    for col in ["Delivery_person_Age", "Delivery_person_Ratings",
                "Vehicle_condition", "multiple_deliveries",
                "Restaurant_latitude", "Restaurant_longitude",
                "Delivery_location_latitude", "Delivery_location_longitude"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["ID", "Delivery_person_ID"])

    print(f"Cleaned: {len(df):,} rows")
    return df

df = clean_df(df)
df.head(3)

Cleaned: 11,399 rows


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City
0,0x2318,COIMBRES13DEL01,NaN,NaN,11.003669,76.976494,11.043669,77.016494,30-03-2022,NaN,15:05:00,NaN,NaN,3,Drinks,electric_scooter,1.0,No,Metropolitian
1,0x3474,BANGRES15DEL01,28.0,4.6,12.975377,77.696664,13.085377,77.806664,29-03-2022,20:30:00,20:35:00,Windy,Jam,0,Snack,motorcycle,1.0,No,Metropolitian
2,0x9420,JAPRES09DEL03,23.0,4.5,26.911378,75.789034,27.001378,75.879034,10-03-2022,19:35:00,19:45:00,Stormy,Jam,0,Drinks,motorcycle,1.0,No,Metropolitian


In [36]:
# Event derivation functions

EVENT_NOTES = {
    "order_placed": ["Order received by system", "Customer confirmed order", None],
    "picked_up":    ["Driver picked up package", "Package collected from restaurant", None],
    "in_transit":   ["Out for delivery", "Driver en route", "Package on the way", None],
    "delivered":    ["Delivered successfully", "Package handed to customer", None],
    "failed":       ["Customer not available", "Wrong address", "Returned to sender"],
}

def jitter(lat, lng, scale=0.005):
    return (round(lat + random.uniform(-scale, scale), 6),
            round(lng + random.uniform(-scale, scale), 6))

def parse_time(t_str, base_date):
    for fmt in ["%H:%M:%S", "%H:%M"]:
        try:
            t = datetime.strptime(str(t_str).strip(), fmt).time()
            return datetime.combine(base_date, t)
        except:
            continue
    return datetime.combine(base_date, dtime(8, 0))

def derive_events(row):
    events = []

    try:
        base_date = datetime.strptime(str(row["Order_Date"]).strip(), "%d-%m-%Y").date()
    except:
        base_date = datetime.today().date()

    rlat = float(row["Restaurant_latitude"]  or 0)
    rlng = float(row["Restaurant_longitude"] or 0)
    dlat = float(row["Delivery_location_latitude"]  or 0)
    dlng = float(row["Delivery_location_longitude"] or 0)

    # No Time_taken(min) in test.csv → estimate randomly
    mins = random.randint(20, 60)

    t_placed    = parse_time(row.get("Time_Orderd"), base_date)
    t_picked    = parse_time(row.get("Time_Order_picked"), base_date)
    t_delivered = t_placed + timedelta(minutes=mins)
    t_transit   = t_picked + (t_delivered - t_picked) / 2

    # Decide final status
    weather = str(row.get("Weatherconditions", "")).lower()
    traffic = str(row.get("Road_traffic_density", "")).lower()
    fail_p  = 0.05
    if "stormy" in weather or "fog" in weather: fail_p += 0.10
    if "jam" in traffic:                        fail_p += 0.05
    final = "failed" if random.random() < fail_p else "delivered"

    # 1. order_placed
    events.append({
        "order_id":   row["ID"],
        "driver_id":  row["Delivery_person_ID"],
        "event_type": "order_placed",
        "timestamp":  t_placed,
        "location":   {"lat": rlat, "lng": rlng},
        "notes":      random.choice(EVENT_NOTES["order_placed"]),
        "metadata": {
            "weather":  row.get("Weatherconditions"),
            "traffic":  row.get("Road_traffic_density"),
            "festival": row.get("Festival"),
            "city":     row.get("City"),
        }
    })

    # 2. picked_up
    pl, pl2 = jitter(rlat, rlng, 0.003)
    events.append({
        "order_id":   row["ID"],
        "driver_id":  row["Delivery_person_ID"],
        "event_type": "picked_up",
        "timestamp":  t_picked,
        "location":   {"lat": pl, "lng": pl2},
        "notes":      random.choice(EVENT_NOTES["picked_up"]),
        "metadata": {
            "vehicle_type":      row.get("Type_of_vehicle"),
            "vehicle_condition": row.get("Vehicle_condition"),
            "driver_rating":     row.get("Delivery_person_Ratings"),
        }
    })

    # 3. in_transit (70% of orders)
    if random.random() > 0.3:
        ml, ml2 = jitter((rlat + dlat) / 2, (rlng + dlng) / 2, 0.005)
        events.append({
            "order_id":   row["ID"],
            "driver_id":  row["Delivery_person_ID"],
            "event_type": "in_transit",
            "timestamp":  t_transit,
            "location":   {"lat": ml, "lng": ml2},
            "notes":      random.choice(EVENT_NOTES["in_transit"]),
            "metadata":   {"estimated_remaining_min": mins // 2}
        })

    # 4. delivered / failed
    fl, fl2 = jitter(dlat, dlng, 0.002)
    events.append({
        "order_id":   row["ID"],
        "driver_id":  row["Delivery_person_ID"],
        "event_type": final,
        "timestamp":  t_delivered,
        "location":   {"lat": fl, "lng": fl2},
        "notes":      random.choice(EVENT_NOTES[final]),
        "metadata":   {"time_taken_min": mins}
    })

    return events

print("Functions ready")

Functions ready


In [ ]:
# Generate all events

all_events = []
for _, row in df.iterrows():
    all_events.extend(derive_events(row.to_dict()))

print(f"{len(all_events):,} events from {len(df):,} orders")
print(f"avg {len(all_events)/len(df):.1f} events per order")

# Preview one document
pprint.pprint(all_events[0])

✅ 42,247 events from 11,399 orders
   avg 3.7 events per order
{'driver_id': 'COIMBRES13DEL01',
 'event_type': 'order_placed',
 'location': {'lat': 11.003669, 'lng': 76.976494},
 'metadata': {'city': 'Metropolitian',
              'festival': 'No',
              'traffic': 'NaN',
              'weather': 'NaN'},
 'notes': None,
 'order_id': '0x2318',
 'timestamp': datetime.datetime(2022, 3, 30, 8, 0)}


In [38]:
# Insert into MongoDB

client = pymongo.MongoClient(MONGO["host"], MONGO["port"])
db     = client[MONGO["database"]]
col    = db[MONGO["collection"]]

col.drop()
print("Inserting...")

BATCH = 5000
inserted = 0
for i in range(0, len(all_events), BATCH):
    col.insert_many(all_events[i : i + BATCH])
    inserted += min(BATCH, len(all_events) - i)
    print(f"  {inserted:,} / {len(all_events):,}", end="\r")

print(f"\nInserted: {col.count_documents({}):,} documents")

Inserting...
  42,247 / 42,247
Inserted: 42,247 documents


In [39]:
# Create Indexes

col.create_index("order_id")
col.create_index("driver_id")
col.create_index("event_type")
col.create_index("timestamp")
col.create_index([("order_id", 1), ("event_type", 1)])

print("Indexes created:")
for idx in col.index_information():
    print(f"  - {idx}")

Indexes created:
  - _id_
  - order_id_1
  - driver_id_1
  - event_type_1
  - timestamp_1
  - order_id_1_event_type_1


In [40]:
# Verify 

print(f"Total documents : {col.count_documents({}):,}")
print(f"Unique orders   : {len(col.distinct('order_id')):,}")

print("\nCount by event_type:")
pipeline = [
    {"$group": {"_id": "$event_type", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}}
]
for doc in col.aggregate(pipeline):
    print(f"  {doc['_id']:<15} {doc['count']:,}")

print("\nSample document (delivered):")
doc = col.find_one({"event_type": "delivered"})
doc.pop("_id", None)
pprint.pprint(doc)

client.close()
print("\nNotebook 3 complete!")

Total documents : 42,247
Unique orders   : 11,399

Count by event_type:
  order_placed    11,399
  picked_up       11,399
  delivered       10,269
  in_transit      8,050
  failed          1,130

Sample document (delivered):
{'driver_id': 'COIMBRES13DEL01',
 'event_type': 'delivered',
 'location': {'lat': 11.044632, 'lng': 77.016675},
 'metadata': {'time_taken_min': 60},
 'notes': None,
 'order_id': '0x2318',
 'timestamp': datetime.datetime(2022, 3, 30, 9, 0)}

Notebook 3 complete!
